In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.6 MB/s eta 0:00:00


In [3]:
!unzip /content/drive/MyDrive/OMR-Datasets/OMR_5Fold_ROIs_split_v4.zip

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_292_1_box48.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam0_1_3_box3.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_217_1_box21.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_308_1_box42.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam1_86_1_box11.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_95_1_box32.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_213_1_box34.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam0_22_3_box9.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_95_1_box21.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_257_1_box57.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_5/val/empty/exam5_201_1_box5.jpg  
  inflating: content/OMR_5Fold_ROIs_spl

In [7]:
import os
len(os.listdir("/content/content/OMR_5Fold_ROIs_split/Fold_1/train_Scen1_withoutGAN/crossedout"))

1500

In [10]:
from ultralytics import YOLO
import os
import numpy as np

K_FOLDS_DIR = "/content/content/OMR_5Fold_ROIs_split"
# chọn kịch bản "train_Scen1_withoutGAN" or "train_Scen2_withGAN"
CHOSEN_SCENARIO = "train_Scen1_withoutGAN" # Đổi tên kịch bản tại đây

fold_yolo_results = []

for fold in range(1, 6):
    print(f"\n{'='*60}")
    print(f"🚀 BẮT ĐẦU HUẤN LUYỆN YOLO26s-cls - FOLD {fold} ({CHOSEN_SCENARIO})")
    print(f"{'='*60}")

    fold_dir = os.path.join(K_FOLDS_DIR, f"Fold_{fold}")

    # ------------------------------------------
    # A. THỦ THUẬT ÉP YOLO ĐỌC ĐÚNG KỊCH BẢN
    # ------------------------------------------
    yolo_train_dir = os.path.join(fold_dir, "train")
    original_train_dir = os.path.join(fold_dir, "train_all_backup")
    scenario_dir = os.path.join(fold_dir, CHOSEN_SCENARIO)

    # Nếu đang có thư mục 'train' thật, đổi tên nó đi để tránh xóa nhầm
    if os.path.exists(yolo_train_dir) and not os.path.islink(yolo_train_dir):
        os.rename(yolo_train_dir, original_train_dir)

    # Tạo symlink (shortcut) tên 'train' trỏ vào thư mục Kịch Bản
    if os.path.islink(yolo_train_dir):
        os.unlink(yolo_train_dir)
    os.symlink(scenario_dir, yolo_train_dir)

    # ------------------------------------------
    # B. KHỞI TẠO MÔ HÌNH VÀ TRAIN
    # ------------------------------------------
    model = YOLO("yolo26s-cls.pt") # Khởi tạo lại weights mới tinh

    results = model.train(
        data=fold_dir,          # YOLO tự động tìm folder 'train' (symlink) và 'val'
        epochs=30,
        imgsz=128,
        batch=128,
        lr0=1e-4,
        patience=10,
        project="/content/drive/MyDrive/OMR-Datasets/train-cls-v2/scene1/Yolo26s",
        name=f"Fold_{fold}",
        # chỉnh TĂNG CƯỜNG ÁNH SÁNG (Photometric) CHO YOLO
        hsv_h=0.01,  # Chỉnh Tone màu nhẹ
        hsv_s=0.3,    # Chỉnh Saturation (Độ bão hòa)
        hsv_v=0.3,    # Chỉnh Sáng/Tối
    )

    # ------------------------------------------
    # C. ĐÁNH GIÁ NGAY TRÊN TẬP TEST (UNSEEN)
    # ------------------------------------------
    print(f"🔍 ĐÁNH GIÁ TẬP TEST FOLD {fold}")
    # YOLO đánh giá thư mục test
    test_results = model.val(data=fold_dir, split='test')

    # Lấy top1_acc từ kết quả
    top1_acc = test_results.top1
    print(f"✅ Accuracy Test Fold {fold}: {top1_acc:.4f}")
    fold_yolo_results.append(top1_acc)

# ==========================================
# 3. TỔNG KẾT BÀI BÁO (YOLO)
# ==========================================
print(f"\n🏆 KẾT QUẢ YOLO26s 5-FOLD ({CHOSEN_SCENARIO}):")
print(f"Danh sách Test Acc: {fold_yolo_results}")
print(f"ĐỘ CHÍNH XÁC TRUNG BÌNH: {np.mean(fold_yolo_results)*100:.2f}% ± {np.std(fold_yolo_results)*100:.2f}%")


🚀 BẮT ĐẦU HUẤN LUYỆN YOLO26s-cls - FOLD 1 (train_Scen1_withoutGAN)
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/content/OMR_5Fold_ROIs_split/Fold_1, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.3, hsv_v=0.3, imgsz=128, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Fold_1-2, nbs